In [3]:
import polars as pl

from jax import numpy as jnp

from summer3.polarized.properties import Property, PropertyTable, LazyExpr
from summer3.polarized.categories import CategoryGroup, CategoryData
from summer3.polarized.flows import FlowSpec, source, dest
from summer3.polarized.expanding import scalar_to_expanding, catdata_to_expanding

pl.Config.set_tbl_rows(64)
pl.Config.set_tbl_width_chars(None)

polars.config.Config

In [4]:
age = Property("age", ["infant", "child","adult", "older"])
state = Property("state", ["S", "I", "R"])
severity = Property("severity", ["mild", "severe"])

pt = PropertyTable.from_property(state).stratify(severity, state=="I").stratify(age, state)

In [5]:
infection = FlowSpec(state == "S", state == "I", pt)

In [6]:
from summer3.managed import ManagedArray, ManagedIndex

In [7]:
def wrap_data_pt(data, pt):
    assert len(data) == len(pt)
    return ManagedArray(data, dims=["compartment"], indices = {"compartment": ManagedIndex("compartment", pt)})

In [47]:
ptd = wrap_data_pt(jnp.linspace(0.0,1.0,len(pt)), pt)

In [54]:
fpt = infection.get_flow_pt()

In [59]:
out_delta = jnp.zeros_like(ptd.data)

In [60]:
out_delta.at[fpt.df["index_dest"].to_numpy()].add(ptd.data[fpt.df["index_source"].to_numpy()])

Array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.06666667, 0.13333333, 0.2       , 0.        , 0.06666667,
       0.13333333, 0.2       , 0.        , 0.        , 0.        ,
       0.        ], dtype=float64)

In [ ]:
ptd.sum((age>="adult") & (state != "S"))

ManagedArray
['compartment'] (6,)
Indices:
['compartment']
Data:
[0.4        0.46666667 0.66666667 0.73333333 0.93333333 1.        ]

In [42]:
ptd[infection.get_flow_pt().df["index_source"]]

TypeError: 'ManagedArray' object is not subscriptable

In [9]:
from datetime import datetime as dt

In [10]:
import numpy as np

In [11]:
import pandas as pd

In [ ]:
x = pd.DatetimeIndex(pd.date_range("1 jan 2000", "30 dec 2000"))[[0,5,6]]

In [27]:
x.to_pydatetime()

array([datetime.datetime(2000, 1, 1, 0, 0),
       datetime.datetime(2000, 1, 6, 0, 0),
       datetime.datetime(2000, 1, 7, 0, 0)], dtype=object)

In [22]:
df = pl.DataFrame({
    "date": pl.datetime_range(dt(2000,1,1),dt(2000,12,31),"1d",eager=True),
    "x": np.arange(366)
})

df.dtypes

[Datetime(time_unit='us', time_zone=None), Int64]

In [34]:
df.filter(pl.col("date").is_in(x.to_pydatetime())

_IncompleteInputError: incomplete input (2322309796.py, line 1)

In [28]:
df.filter(pl.col('date').is_in(x.to_pydatetime()))

date,x
datetime[μs],i64
2000-01-01 00:00:00,0
2000-01-06 00:00:00,5
2000-01-07 00:00:00,6


In [23]:
pd.DataFrame(index=x)

""
2000-01-01
2000-01-06
2000-01-07


In [25]:
df

date,x
datetime[μs],i64
2000-01-01 00:00:00,0
2000-01-02 00:00:00,1
2000-01-03 00:00:00,2
2000-01-04 00:00:00,3
2000-01-05 00:00:00,4
2000-01-06 00:00:00,5
2000-01-07 00:00:00,6
2000-01-08 00:00:00,7
2000-01-09 00:00:00,8


In [24]:
df.filter(pl.col('date').is_in(pl.from_pandas(x)))

C:\Users\dshi0012\AppData\Local\Temp\ipykernel_1732\465897875.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df.filter(pl.col('date').is_in(pl.from_pandas(x)))


InvalidOperationError: 'is_in' cannot check for Nanoseconds precision values in Microseconds Datetime data

In [18]:
pl.from_pandas(x)#pd.DataFrame(index=x),include_index=True)

""
datetime[ns]
2000-01-01 00:00:00
2000-01-06 00:00:00
2000-01-07 00:00:00


In [12]:
fpt = infection.get_flow_pt()

In [14]:
ma = ManagedArray(jnp.linspace(0.0,1.0,len(fpt)), dims=["flow"], indices = {"flow": ManagedIndex("flow", fpt)})

In [15]:
ma.indices

{'flow': ManagedIndex: maps flow
 PropertyTable
 shape: (8, 9)
 ┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
 │ state_0_so ┆ severity_0 ┆ age_0_sour ┆ index_sou ┆ … ┆ severity_ ┆ age_0_des ┆ index_des ┆ index │
 │ urce       ┆ _source    ┆ ce         ┆ rce       ┆   ┆ 0_dest    ┆ t         ┆ t         ┆ ---   │
 │ ---        ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ i64   │
 │ str        ┆ str        ┆ str        ┆ i64       ┆   ┆ str       ┆ str       ┆ i64       ┆       │
 ╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
 │ S          ┆ null       ┆ infant     ┆ 0         ┆ … ┆ mild      ┆ infant    ┆ 4         ┆ 0     │
 │ S          ┆ null       ┆ child      ┆ 1         ┆ … ┆ mild      ┆ child     ┆ 5         ┆ 1     │
 │ S          ┆ null       ┆ adult      ┆ 2         ┆ … ┆ mild      ┆ adult     ┆ 6         ┆ 2     │
 │ S          ┆ nul

In [27]:
age.traits

array(['infant', 'child', 'adult', 'older'], dtype='<U6')

In [59]:
le = ((dest(age)=="adult") & (dest(age)=="infant"))

In [42]:
fpt.filter((dest(age)=="adult") & (dest(age)=="infant"))

state_0_source,severity_0_source,age_0_source,index_source,state_0_dest,severity_0_dest,age_0_dest,index_dest,index
str,str,str,i64,str,str,str,i64,i64


In [63]:
ma.query(
    (dest(age)>="child") & (source(age)=="adult")
).indices

{'flow': ManagedIndex: maps flow
 PropertyTable
 shape: (2, 9)
 ┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
 │ state_0_so ┆ severity_0 ┆ age_0_sour ┆ index_sou ┆ … ┆ severity_ ┆ age_0_des ┆ index_des ┆ index │
 │ urce       ┆ _source    ┆ ce         ┆ rce       ┆   ┆ 0_dest    ┆ t         ┆ t         ┆ ---   │
 │ ---        ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ i64   │
 │ str        ┆ str        ┆ str        ┆ i64       ┆   ┆ str       ┆ str       ┆ i64       ┆       │
 ╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
 │ S          ┆ null       ┆ adult      ┆ 2         ┆ … ┆ mild      ┆ adult     ┆ 6         ┆ 0     │
 │ S          ┆ null       ┆ adult      ┆ 2         ┆ … ┆ severe    ┆ adult     ┆ 10        ┆ 1     │
 └────────────┴────────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────┘}

In [32]:
ma.data[ma.query(
    (dest(age)>="child")
).parent_indices]

Array([0.14285714, 0.28571429, 0.42857143, 0.71428571, 0.85714286,
       1.        ], dtype=float64)

In [20]:
ma.query(dest(age)=="infant").parent_indices

slice(np.int64(0), np.int64(8), np.int64(4))

In [23]:
dest(severity).categories()

CategoryGroup:
{'dest(severity)_mild': Property[dest(severity)(None)] == mild, 'dest(severity)_severe': Property[dest(severity)(None)] == severe}

In [24]:
severity_split = CategoryData(dest(severity).categories(), jnp.array([0.8,0.2]))
age_suscept = CategoryData(source(age)[["adult","older"]].categories(), jnp.array([1.2,1.8]))

In [25]:
catdata_to_expanding(age_suscept, fpt)

ValueError: cat_data must cover entire PropertyTable

In [16]:
fpt

state_0_source,severity_0_source,age_0_source,index_source,state_0_dest,severity_0_dest,age_0_dest,index_dest,index
str,str,str,i64,str,str,str,i64,i64
"""S""",null,"""infant""",0,"""I""","""mild""","""infant""",4,0
"""S""",null,"""child""",1,"""I""","""mild""","""child""",5,1
"""S""",null,"""adult""",2,"""I""","""mild""","""adult""",6,2
"""S""",null,"""older""",3,"""I""","""mild""","""older""",7,3
"""S""",null,"""infant""",0,"""I""","""severe""","""infant""",8,4
"""S""",null,"""child""",1,"""I""","""severe""","""child""",9,5
"""S""",null,"""adult""",2,"""I""","""severe""","""adult""",10,6
"""S""",null,"""older""",3,"""I""","""severe""","""older""",11,7


In [18]:
cat

CategoryGroup:
{'source(age)_adult': Property[source(age)(None)] == adult, 'source(age)_older': Property[source(age)(None)] == older}

In [15]:
base_param = scalar_to_expanding(1.0, fpt)
base_param.apply_op(severity_split).apply_op(age_suscept)

ExpandingArray
shape: (8,)
Series: 'opidx' [i64]
[
	4
	4
	2
	0
	1
	1
	3
	5
]
[1.44 0.8  0.96 0.96 0.8  1.44]
[0.8  0.8  0.96 1.44 0.8  0.8  0.96 1.44]

In [ ]:
base_param.apply_op()

Array([1., 1., 1., 1., 1., 1., 1., 1.], dtype=float64)

In [48]:
eacd.data.sha

(4,)

In [47]:
.apply_op(severity_split)

ExpandingArray
shape: (8,)
Series: 'opidx' [i64]
[
	0
	1
	2
	3
	4
	5
	6
	7
]
[0.8 0.8 0.8 0.2 0.2 0.2 0.2 0.8]

In [43]:
catdata_to_stacked(source(age).categories(), jnp.ones(4))

AttributeError: 'dict' object has no attribute 'exclusive'

In [52]:
scalar_to_expanding(1.0, fpt).apply_op(age_suscept)

ExpandingArray
shape: (8,)
Series: 'opidx' [i64]
[
	0
	0
	1
	2
	0
	0
	1
	2
]
[1.  1.2 1.8]

In [40]:
from jax import jit, make_jaxpr

In [41]:
@jit
def do_things(sev_data, age_data):
    sd = orig_sd

    severity_split = PStackedOp(dest(severity).categories(), sev_data)
    age_suscept = PStackedOp(source(age)[["adult","older"]].categories(), age_data)

    sd = apply_pop(sd, severity_split, fpt)
    sd = apply_pop(orig_sd, age_suscept, fpt)
    
    return sd.data[sd.df.to_numpy()[0]]

In [ ]:


#op0 = StackedOp({0: (2,0), 1: (2,2), 2: (2,4), 3: (2,7), 4: (2,9)}, jnp.array([2.0,3.0,5.0,7.0,9.0]))
#op1 = StackedOp({0: (0,1), 1:(0,2), 2:(0,3)}, jnp.array([5.0,7.0,11.0]))
#severity_split = StackedOp({0: (1,0), 1: (1,1), 2: (1,2)}, jnp.array([0.3,0.5,0.2]))

In [34]:
apply_op(orig_sd, severity_split).data

NameError: name 'orig_sd' is not defined

In [278]:
sd0 = apply_op(orig_sd, op0)

In [280]:
apply_op(sd0, op1)

KeyboardInterrupt: 

In [273]:
apply_op(orig_sd, op0).apply_op(op1).data

Array([49., 99.,  3., 11.,  7., 25.,  5.,  1.,  2.], dtype=float32)

In [276]:
apply_op(orig_sd, op1).apply_op(op0).data

Array([11.,  1., 25.,  3., 49.,  5., 99.,  7.,  2.], dtype=float32)

In [241]:
apply_op(orig_sd, op0).data

Array([1., 3., 2.], dtype=float32)

In [207]:
apply_op(orig_df, op0)

column_0,column_1
i64,i64
0,1
0,0
0,null


In [ ]:
unique_opdf = df[0:2].transpose(include_header=True).drop("column").unique()
sorted_by_lastop = unique_opdf.sort("column_1")
print(sorted_by_lastop)
null_mask = sorted_by_lastop.with_columns(pl.all().is_null())
rhs_mask = ~null_mask["column_1"]

lhs_indices = sorted_by_lastop["column_0"].to_numpy()
# The indices of the RHS operator we are updating with
rhs_selector = sorted_by_lastop["column_1"].filter(rhs_mask).to_numpy()



# These are the 'at' selectors for the expanded LHS (ie where to update)
active_indices_lhs = np.arange(len(unique_opdf))[rhs_mask]

#active_indices_lhs, rhs_selector

orig[lhs_indices].at[active_indices_lhs].mul(oparr1[rhs_selector])

In [158]:
dft = df[0:2].transpose()
dft.with_columns(pl.struct("column_0","column_1").alias("group"))

column_0,column_1,group
i64,i64,struct[2]
0,0,"{0,0}"
0,null,"{0,null}"
0,1,"{0,1}"
0,0,"{0,0}"
0,null,"{0,null}"
…,…,…
0,null,"{0,null}"
0,1,"{0,1}"
0,0,"{0,0}"


In [153]:
sorted_by_lastop.with_columns(pl.struct("column_0","column_1").alias("group"))

column_0,column_1,group
i64,i64,struct[2]
0,null,"{0,null}"
0,0,"{0,0}"
0,1,"{0,1}"


In [154]:
# Starting from a scalar, we now have a 3 element
# arr, with 2 muls and one 'passthrough'
# sc

unique_opdf = df[0:2].transpose(include_header=True).drop("column").unique()
sorted_by_lastop = unique_opdf.sort("column_1")
print(sorted_by_lastop)
null_mask = sorted_by_lastop.with_columns(pl.all().is_null())
rhs_mask = ~null_mask["column_1"]

lhs_indices = sorted_by_lastop["column_0"].to_numpy()
# The indices of the RHS operator we are updating with
rhs_selector = sorted_by_lastop["column_1"].filter(rhs_mask).to_numpy()



# These are the 'at' selectors for the expanded LHS (ie where to update)
active_indices_lhs = np.arange(len(unique_opdf))[rhs_mask]

#active_indices_lhs, rhs_selector

orig[lhs_indices].at[active_indices_lhs].mul(oparr1[rhs_selector])

shape: (3, 2)
┌──────────┬──────────┐
│ column_0 ┆ column_1 │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 0        ┆ null     │
│ 0        ┆ 0        │
│ 0        ┆ 1        │
└──────────┴──────────┘


Array([2. , 3. , 3.4], dtype=float32)

In [163]:
df.transpose()

column_0,column_1,column_2
i64,i64,i64
0,0,null
0,null,null
0,1,null
0,0,0
0,null,0
…,…,…
0,null,1
0,1,1
0,0,2


In [175]:
gwi = sorted_by_lastop.with_columns(
    pl.Series("opidx",np.array((0,1,2))),pl.struct("column_0","column_1").alias("op_group")
    ).drop("column_0","column_1")
gwi

opidx,op_group
i64,struct[2]
0,"{0,null}"
1,"{0,0}"
2,"{0,1}"


In [176]:
df.transpose().with_columns(pl.struct("column_0","column_1").alias("op_group")).join(gwi, on="op_group")

column_0,column_1,column_2,op_group,opidx
i64,i64,i64,struct[2],i64
0,0,null,"{0,0}",1
0,null,null,"{0,null}",0
0,1,null,"{0,1}",2
0,0,0,"{0,0}",1
0,null,0,"{0,null}",0
…,…,…,…,…
0,null,1,"{0,null}",0
0,1,1,"{0,1}",2
0,0,2,"{0,0}",1
